# Smoke Pipeline Run (Fast)

A short end-to-end check of the full SFT -> DPO flow.

Use this notebook to validate that scripts, paths, and environment work before a long run.

## 0) Environment setup (one-time)

If this machine is already prepared, skip this cell.

In [ ]:
import sys
print(sys.executable)

In [ ]:
from pathlib import Path
import subprocess
import shlex
from datetime import datetime

ROOT = Path.cwd()
print("Workspace:", ROOT)
assert (ROOT / "scripts").exists(), "Run notebook from /workspace/thesis-llm-alignment"

In [ ]:
# Smoke config (small and quick)
BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATASET_ID = "tatsu-lab/alpaca"

RAW_DPO_PAIRS = "data/dpo_pairs.smoke.raw.jsonl"
CLEAN_DPO_PAIRS = "data/dpo_pairs.smoke.clean.jsonl"
CLEAN_STATS = "logs/dpo_pairs_smoke_stats.json"

SFT_OUT = "outputs/qwen2.5-3b-sft-lora-smoke"
DPO_OUT = "outputs/qwen2.5-3b-sft-dpo-lora-smoke"
COMPARE_OUT = "results/compare_outputs_smoke.jsonl"

NUM_EXAMPLES_FOR_DPO = 64
MAX_STEPS_SFT = 20
MAX_STEPS_DPO = 20
BATCH_SIZE = 1
GRAD_ACCUM = 8
MAX_SEQ_LEN = 768
MAX_PROMPT_LEN_DPO = 384
LR_SFT = 2e-4
LR_DPO = 1e-5
BETA_DPO = 0.1
SEED = 42
PROMPT_STYLE = "chat"

GEN_MAX_NEW_TOKENS_BUILD = 96
GEN_MAX_NEW_TOKENS_COMPARE = 120
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.9

SMOKE_PROMPTS = [
    "Explain RLHF in simple terms.",
    "Give 3 bullet points on why unit tests matter.",
    "Translate to English: 'Мне нужно подготовить отчет по экспериментам.'",
    "What is overfitting and how can we reduce it?",
]

print("Smoke config loaded")

In [ ]:
def run(cmd: str):
    print("\n$", cmd)
    subprocess.run(cmd, shell=True, check=True)

def q(x):
    return shlex.quote(str(x))

Path("logs").mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

## 1) Build raw DPO pairs

In [ ]:
cmd_build_pairs = (
    "python scripts/build_dpo_pairs.py "
    f"--base_model_id {q(BASE_MODEL_ID)} "
    f"--dataset_id {q(DATASET_ID)} "
    f"--out_path {q(RAW_DPO_PAIRS)} "
    f"--num_examples {NUM_EXAMPLES_FOR_DPO} "
    f"--max_new_tokens {GEN_MAX_NEW_TOKENS_BUILD} "
    f"--temperature {GEN_TEMPERATURE} "
    f"--top_p {GEN_TOP_P} "
    f"--prompt_style {q(PROMPT_STYLE)}"
)
run(cmd_build_pairs)

## 2) Clean pairs

In [ ]:
cmd_clean_pairs = (
    "python scripts/clean_dpo_pairs_full.py "
    f"--prompt_style {q(PROMPT_STYLE)} "
    f"--in_path {q(RAW_DPO_PAIRS)} "
    f"--out_path {q(CLEAN_DPO_PAIRS)} "
    "--min_chars 20 "
    "--drop_if_truncated "
    "--drop_if_equal "
    "--make_strict_prompt "
    f"--stats_path {q(CLEAN_STATS)}"
)
run(cmd_clean_pairs)

## 3) Train SFT

In [ ]:
cmd_train_sft = (
    "python scripts/train_sft.py "
    f"--model_id {q(BASE_MODEL_ID)} "
    f"--dataset_id {q(DATASET_ID)} "
    f"--output_dir {q(SFT_OUT)} "
    f"--max_steps {MAX_STEPS_SFT} "
    f"--lr {LR_SFT} "
    f"--batch_size {BATCH_SIZE} "
    f"--grad_accum {GRAD_ACCUM} "
    f"--max_seq_len {MAX_SEQ_LEN} "
    f"--seed {SEED} "
    f"--prompt_style {q(PROMPT_STYLE)}"
)
run(cmd_train_sft)

## 4) Train DPO

In [ ]:
cmd_train_dpo = (
    "python scripts/train_dpo.py "
    f"--base_model_id {q(BASE_MODEL_ID)} "
    f"--sft_adapter_dir {q(SFT_OUT)} "
    f"--dpo_data_path {q(CLEAN_DPO_PAIRS)} "
    f"--output_dir {q(DPO_OUT)} "
    f"--max_steps {MAX_STEPS_DPO} "
    f"--batch_size {BATCH_SIZE} "
    f"--grad_accum {GRAD_ACCUM} "
    f"--max_length {MAX_SEQ_LEN} "
    f"--max_prompt_length {MAX_PROMPT_LEN_DPO} "
    f"--lr {LR_DPO} "
    f"--beta {BETA_DPO} "
    f"--seed {SEED}"
)
run(cmd_train_dpo)

## 5) Compare outputs on a small prompt set

In [ ]:
tmp_prompts_path = Path("data/compare_prompts.smoke.txt")
tmp_prompts_path.write_text("\n".join(SMOKE_PROMPTS) + "\n", encoding="utf-8")
print("Saved prompts to", tmp_prompts_path)

In [ ]:
cmd_compare = (
    "python scripts/compare_models.py "
    f"--base_model_id {q(BASE_MODEL_ID)} "
    f"--sft_adapter_dir {q(SFT_OUT)} "
    f"--dpo_adapter_dir {q(DPO_OUT)} "
    f"--prompts_path {q(tmp_prompts_path)} "
    f"--out_path {q(COMPARE_OUT)} "
    f"--max_new_tokens {GEN_MAX_NEW_TOKENS_COMPARE} "
    f"--temperature {GEN_TEMPERATURE} "
    f"--top_p {GEN_TOP_P} "
    "--do_sample"
)
run(cmd_compare)

In [ ]:
print("\nSmoke pipeline finished.")
print("Artifacts:")
print("-", RAW_DPO_PAIRS)
print("-", CLEAN_DPO_PAIRS)
print("-", CLEAN_STATS)
print("-", SFT_OUT)
print("-", DPO_OUT)
print("-", COMPARE_OUT)
print("Completed at:", datetime.now().isoformat(timespec="seconds"))